In [ ]:
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer
import re
import pandas as pd

# Veri yükle ve temizle
dataset = load_dataset("truehealth/medicationqa")

def remove_empty(example):
    return (
        example["Question"] is not None
        and example["Answer"] is not None
        and len(example["Question"].strip()) > 0
        and len(example["Answer"].strip()) > 0
    )

dataset["train"] = dataset["train"].filter(remove_empty)

def clean_text(example):
    question = re.sub(r"\s+", " ", example["Question"]).strip()
    answer = re.sub(r"\s+", " ", example["Answer"]).strip()
    return {"Question": question, "Answer": answer}

dataset["train"] = dataset["train"].map(clean_text)

df = dataset["train"].to_pandas()
df = df.drop_duplicates(subset=["Question", "Answer"])

# Split
clean_dataset = Dataset.from_pandas(df)
train_test = clean_dataset.train_test_split(test_size=0.2, seed=42)
validation_test = train_test["test"].train_test_split(test_size=0.5, seed=42)

final_dataset = {
    "train": train_test["train"],
    "validation": validation_test["train"],
    "test": validation_test["test"]
}

print(f"Train: {len(final_dataset['train'])}")
print(f"Validation: {len(final_dataset['validation'])}")
print(f"Test: {len(final_dataset['test'])}")

# Tokenizer ve model
distilbert_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
distilbert_model = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-uncased")

# Preprocess — context olarak Answer sütununu kullanıyoruz
def preprocess_function(examples):
    tokenized = distilbert_tokenizer(
        examples["Question"],
        examples["Answer"],   # ← context = answer metni
        truncation=True,
        padding="max_length",
        max_length=512,
        return_offsets_mapping=True
    )

    start_positions = []
    end_positions = []

    for i in range(len(examples["Answer"])):
        context = examples["Answer"][i]

        # Cevabın ilk cümlesini hedef al
        first_sentence = context.split(".")[0].strip()
        if len(first_sentence) < 5:
            first_sentence = context[:100]

        start_char = context.find(first_sentence)
        if start_char == -1:
            start_char = 0
        end_char = start_char + len(first_sentence)

        offsets = tokenized["offset_mapping"][i]
        sequence_ids = tokenized.sequence_ids(i)

        start_token = 0
        end_token = 0

        for idx, (offset, seq_id) in enumerate(zip(offsets, sequence_ids)):
            if seq_id != 1:  # sadece context tokenları
                continue
            start, end = offset
            if start <= start_char < end:
                start_token = idx
            if start < end_char <= end:
                end_token = idx

        start_positions.append(start_token)
        end_positions.append(end_token)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    tokenized.pop("offset_mapping")

    return tokenized

# Tokenize et
print("Tokenize ediliyor...")
tokenized_train = final_dataset["train"].map(preprocess_function, batched=True)
tokenized_validation = final_dataset["validation"].map(preprocess_function, batched=True)
tokenized_test = final_dataset["test"].map(preprocess_function, batched=True)

# Gereksiz sütunları kaldır
columns_to_remove = ["Question", "Answer", "Focus (Drug)",
                     "Question Type", "Section Title", "URL"]

# __index_level_0__ varsa ekle
if "__index_level_0__" in tokenized_train.column_names:
    columns_to_remove.append("__index_level_0__")

tokenized_train = tokenized_train.remove_columns(columns_to_remove)
tokenized_validation = tokenized_validation.remove_columns(columns_to_remove)

print("Hazır ✓")
print("Sütunlar:", tokenized_train.column_names)

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./distilbert_results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    fp16=True
)

trainer = Trainer(
    model=distilbert_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
)

print("DistilBERT eğitimi başlıyor...")
trainer.train()
print("Eğitim tamamlandı ✓")

distilbert_model.save_pretrained("./distilbert_final")
distilbert_tokenizer.save_pretrained("./distilbert_final")
print("Model kaydedildi ✓")

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
import pandas as pd

# test.csv yeniden oluştur
dataset = load_dataset("truehealth/medicationqa")
df_raw = dataset["train"].to_pandas()
df_raw = df_raw.rename(columns={"Question": "question", "Answer": "answer"})
df_raw = df_raw[["question", "answer"]].dropna()

train_df, temp_df = train_test_split(df_raw, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv", index=False)
test_df.to_csv("test.csv", index=False)
print("CSV dosyaları oluşturuldu ✓")

# Kaydedilen DistilBERT modelini yükle
distilbert_tokenizer = AutoTokenizer.from_pretrained("./distilbert_final")
distilbert_model = AutoModelForQuestionAnswering.from_pretrained("./distilbert_final")
print("DistilBERT yüklendi ✓")

# Test et
qa_pipeline = pipeline(
    "question-answering",
    model=distilbert_model,
    tokenizer=distilbert_tokenizer
)

for i in range(3):
    soru = test_df["question"].iloc[i]
    context = test_df["answer"].iloc[i]
    gercek = test_df["answer"].iloc[i]

    sonuc = qa_pipeline(
        question=soru,
        context=context,
        max_answer_len=200,
        top_k=1
    )

    print(f"Soru: {soru}")
    print(f"Gerçek Cevap: {gercek[:200]}...")
    print(f"DistilBERT Cevabı: {sonuc['answer']}")
    print(f"Güven Skoru: {sonuc['score']:.4f}")
    print("-" * 60)

In [ ]:
!pip install gradio -q

In [ ]:
import gradio as gr

def distilbert_chat(user_question, history):
    query_embedding = embedder.encode(user_question, convert_to_tensor=True)
    scores = util.cos_sim(query_embedding, question_embeddings)[0]
    best_idx = scores.argmax().item()
    best_score = scores[best_idx].item()

    if best_score < 0.5:
        cevap = "Bu soruya uygun bir cevap bulunamadı. Lütfen sorunuzu farklı şekilde sorun."
        bilgi = "-"
        bulunan = "-"
    else:
        context = train_df["answer"].iloc[best_idx]
        bulunan_soru = train_df["question"].iloc[best_idx]

        sonuc = qa_pipeline(
            question=user_question,
            context=context,
            max_answer_len=200,
            top_k=1
        )

        cevap = sonuc["answer"]
        bilgi = f"Benzerlik Skoru: {best_score:.3f} | Güven Skoru: {sonuc['score']:.4f}"
        bulunan = bulunan_soru

    history.append((user_question, cevap))
    return history, bilgi, bulunan, ""

with gr.Blocks(title="İlaç Bilgilendirme Chatbot — DistilBERT") as demo:
    gr.Markdown("# 💊 İlaç Bilgilendirme Chatbot — DistilBERT")
    gr.Markdown("İlaçlarla ilgili sorularınızı İngilizce olarak sorun.")

    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Sohbet Geçmişi", height=400)
            user_input = gr.Textbox(
                placeholder="İlaç sorunuzu buraya yazın...",
                label="Sorunuz",
                lines=2
            )
            with gr.Row():
                submit_btn = gr.Button("Gönder", variant="primary")
                clear_btn = gr.Button("Temizle")

        with gr.Column(scale=1):
            skorlar = gr.Textbox(label="Skorlar", lines=2, interactive=False)
            bulunan_soru = gr.Textbox(label="Eşleşen Soru", lines=2, interactive=False)

    gr.Examples(
        examples=[
            ["what is ibuprofen used for?"],
            ["what are the side effects of aspirin?"],
            ["how is tetracycline metabolized?"],
            ["what is metformin used for?"]
        ],
        inputs=user_input
    )

    history_state = gr.State([])

    submit_btn.click(
        fn=distilbert_chat,
        inputs=[user_input, history_state],
        outputs=[chatbot, skorlar, bulunan_soru, user_input]
    ).then(lambda h: h, history_state, history_state)

    clear_btn.click(lambda: ([], [], "", ""), outputs=[chatbot, history_state, skorlar, bulunan_soru])

demo.launch(share=True)

In [ ]:
!pip install evaluate rouge_score -q

In [ ]:
import pandas as pd
from evaluate import load
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline, AutoTokenizer, AutoModelForQuestionAnswering
import torch

# Test verisini yükle
test_df = pd.read_csv("test.csv")
train_df = pd.read_csv("train.csv")

# Sentence-BERT modelini yükle
embedder = SentenceTransformer('distilbert-base-nli-stsb-mean-tokens')

# Eğitim setindeki soruların embeddinglerini oluştur
question_embeddings = embedder.encode(train_df["question"].tolist(), convert_to_tensor=True)

# Metrikler
rouge = load("rouge")
bleu = load("bleu")

predictions = []
references = []

print("Test seti üzerinde tahminler üretiliyor...")

for _, row in test_df.iterrows():
    soru = row["question"]
    gercek_cevap = row["answer"]

    # Sentence-BERT ile context bul
    query_embedding = embedder.encode(soru, convert_to_tensor=True)
    scores = util.cos_sim(query_embedding, question_embeddings)[0]
    best_idx = scores.argmax().item()
    best_score = scores[best_idx].item()

    if best_score < 0.5:
        predictions.append("no answer found")
    else:
        context = train_df["answer"].iloc[best_idx]
        sonuc = qa_pipeline(
            question=soru,
            context=context,
            max_answer_len=200,
            top_k=1
        )
        predictions.append(sonuc["answer"])

    references.append(gercek_cevap)

print(f"Toplam tahmin: {len(predictions)}")

# ROUGE
rouge_scores = rouge.compute(predictions=predictions, references=references)

# BLEU
bleu_score = bleu.compute(
    predictions=predictions,
    references=[[r] for r in references]
)

# Exact Match
exact_match = sum(p.strip().lower() == r.strip().lower()
                  for p, r in zip(predictions, references)) / len(predictions)

# F1
def compute_f1(pred, ref):
    pred_tokens = pred.lower().split()
    ref_tokens = ref.lower().split()
    common = set(pred_tokens) & set(ref_tokens)
    if not common:
        return 0.0
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

f1_scores = [compute_f1(p, r) for p, r in zip(predictions, references)]
avg_f1 = sum(f1_scores) / len(f1_scores)

print("\n=== DistilBERT Değerlendirme Sonuçları ===")
print(f"ROUGE-1:      {rouge_scores['rouge1']:.4f}")
print(f"ROUGE-2:      {rouge_scores['rouge2']:.4f}")
print(f"ROUGE-L:      {rouge_scores['rougeL']:.4f}")
print(f"BLEU:         {bleu_score['bleu']:.4f}")
print(f"Exact Match:  {exact_match:.4f}")
print(f"F1 Score:     {avg_f1:.4f}")

# Sonuçları kaydet
results_df = pd.DataFrame({
    "question": test_df["question"],
    "reference": references,
    "distilbert_prediction": predictions
})
results_df.to_csv("distilbert_results.csv", index=False)
print("\nSonuçlar kaydedildi ✓")